In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import mhnlib.utils as mhn_utils
from math import log, sqrt
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from typing import Optional, Tuple, Union
from pathlib import Path
import seaborn as sns
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

# IID patterns

## Generate data (dual)

In [ ]:
torch.manual_seed(1101252)
N = 16
K_max = 256
Ks = torch.arange(16,K_max+1, step=8)
patterns_per_K = {}
for K in Ks:
    S = max(3*K, 128)
    patterns_per_K[K.item()] = torch.randn(S, K.item(), N)/sqrt(N)

### Quenched (dual)

In [ ]:
overwrite = False
torch.manual_seed(1101252)

num_betas = 100
num_iterations = 10_000
num_ics_per_batch = N
num_batches = 3
num_ics = num_batches * num_ics_per_batch

logit_noise_std = 1 / sqrt(N)
betas_quench = torch.logspace(-1, 2, steps=num_betas)

for K, patterns in patterns_per_K.items():
    print(f"Processing K={K}...")

    filename = (
        f"paper_results/local_data/"
        f"iid_weights_quenched_N={N}_K={K}.npz"
    )

    if Path(filename).exists() and not overwrite:
        print(f"File {filename} already exists, skipping...")
        continue

    weights_ic = torch.softmax(
        torch.randn(num_ics, K) * logit_noise_std,
        dim=-1,
    )

    results = {}

    for is_centered in [False, True]:
        for is_euclidean in [False, True]:

            # Centered and uncentered Euclidean models are equivalent
            # under a common translation of all patterns.
            if is_euclidean and is_centered:
                continue

            if device.type == "mps":
                torch.mps.empty_cache()
            elif device.type == "cuda":
                torch.cuda.empty_cache()

            patterns_used = (
                patterns - patterns.mean(dim=-2, keepdim=True)
                if is_centered
                else patterns
            )

            grams = mhn_utils.get_gram_matrix(patterns_used)

            diagonal = torch.diagonal(
                grams,
                dim1=-2,
                dim2=-1,
            )

            if is_euclidean:
                biases = -0.5 * diagonal
            else:
                biases = torch.zeros_like(diagonal)

            w_quench_fps = []

            for batch_idx in tqdm(
                range(num_batches),
                desc=(
                    f"K={K}, centered={is_centered}, "
                    f"euclidean={is_euclidean}"
                ),
            ):
                batch_start = batch_idx * num_ics_per_batch
                batch_end = (batch_idx + 1) * num_ics_per_batch

                w_batch = mhn_utils.dual_deterministic_dynamics(
                    grams.to(device),
                    biases.to(device),
                    betas_quench.to(device),
                    weights_ic[batch_start:batch_end].to(device),
                    num_iterations,
                    verbose=False,
                )

                w_quench_fps.append(w_batch.detach().cpu())

            w_quench_fps = torch.cat(w_quench_fps, dim=-2)
            entropies = mhn_utils.get_entropy(w_quench_fps)

            suffix = (
                f"centered_{is_centered}_"
                f"euclidean_{is_euclidean}"
            )

            results[f"probs_{suffix}"] = w_quench_fps.numpy()
            results[f"entropies_{suffix}"] = entropies.numpy()

    np.savez(
        filename,
        patterns=patterns.detach().cpu().numpy(),
        betas=betas_quench.numpy(),
        **results,
    )

### Annealed (dual)

In [ ]:
overwrite = False
torch.manual_seed(1101252)

num_betas = 100
num_iterations = 10_000
num_ics_per_batch = N
num_batches = 3
num_ics = num_batches * num_ics_per_batch

logit_noise_std = 1 / sqrt(N)
betas_anneal = torch.logspace(-1, 2, steps=num_betas)

for K, patterns in patterns_per_K.items():
    print(f"Processing K={K}...")

    filename = (
        f"paper_results/local_data/"
        f"iid_weights_annealed_N={N}_K={K}.npz"
    )

    if Path(filename).exists() and not overwrite:
        print(f"File {filename} already exists, skipping...")
        continue

    weights_ic = torch.softmax(
        torch.randn(num_ics, K) * logit_noise_std,
        dim=-1,
    )

    results = {}

    for is_centered in [False, True]:
        for is_euclidean in [False, True]:

            # Centered and uncentered Euclidean models are equivalent
            # under a common translation of all patterns.
            if is_euclidean and is_centered:
                continue

            if device.type == "mps":
                torch.mps.empty_cache()
            elif device.type == "cuda":
                torch.cuda.empty_cache()

            patterns_used = (
                patterns - patterns.mean(dim=-2, keepdim=True)
                if is_centered
                else patterns
            )

            grams = mhn_utils.get_gram_matrix(patterns_used)

            diagonal = torch.diagonal(
                grams,
                dim1=-2,
                dim2=-1,
            )

            if is_euclidean:
                biases = -0.5 * diagonal
            else:
                biases = torch.zeros_like(diagonal)

        
            w_anneal_fps = mhn_utils.dual_deterministic_dynamics_annealing(grams.to(device),
                                                                biases.to(device),
                                                                betas_anneal.to(device),
                                                                weights_ic.to(device),
                                                                logit_noise_std=logit_noise_std,
                                                                num_iterations=num_iterations,
                                                                verbose=True).cpu()


            entropies = mhn_utils.get_entropy(w_anneal_fps)

            suffix = (
                f"centered_{is_centered}_"
                f"euclidean_{is_euclidean}"
            )

            results[f"probs_{suffix}"] = w_anneal_fps.numpy()
            results[f"entropies_{suffix}"] = entropies.numpy()

    np.savez(
        filename,
        patterns=patterns.detach().cpu().numpy(),
        betas=betas_quench.numpy(),
        **results,
    )

## Plots quenched (dual)

In [ ]:
data_folder = Path('paper_results/local_data/')
N = 16
betas_quenched_per_K = {}
entropies_quenched_centered_per_K = {}
entropies_quenched_non_centered_per_K = {}
entropies_quenched_euclidean_per_K = {}
patterns_quenched_per_K = {}
betas_quenched = None
for quenched_file in tqdm(list(data_folder.glob(f'iid_weights_quenched_N={N}_K=*.npz')), desc='Loading data...'):
    data = np.load(quenched_file, allow_pickle=True)
    K = int(quenched_file.stem.split('K=')[-1])
    entropies_quenched_centered_per_K[K] = torch.as_tensor(data['entropies_centered_True_euclidean_False'])
    entropies_quenched_non_centered_per_K[K] = torch.as_tensor(data['entropies_centered_False_euclidean_False'])
    entropies_quenched_euclidean_per_K[K] = torch.as_tensor(data['entropies_centered_False_euclidean_True'])
    betas_quenched = torch.as_tensor(data['betas'])
    patterns_quenched_per_K[K] = torch.as_tensor(data['patterns'])

In [ ]:
all_Ks = list(sorted(entropies_quenched_centered_per_K.keys()))
mean_entropies_quenched_centered = []
mean_entropies_quenched_non_centered = []
mean_entropies_quenched_euclidean = []
for K in all_Ks:
    mean_entropies_quenched_centered.append(entropies_quenched_centered_per_K[K].mean(dim=(1,2)))
    mean_entropies_quenched_non_centered.append(entropies_quenched_non_centered_per_K[K].mean(dim=(1,2)))
    mean_entropies_quenched_euclidean.append(entropies_quenched_euclidean_per_K[K].mean(dim=(1,2)))
mean_entropies_quenched_centered = torch.stack(mean_entropies_quenched_centered)
mean_entropies_quenched_non_centered = torch.stack(mean_entropies_quenched_non_centered)
mean_entropies_quenched_euclidean = torch.stack(mean_entropies_quenched_euclidean)
Ks = torch.tensor(all_Ks)

In [ ]:
beta_c_mp = Ks/(1+torch.sqrt(Ks/N))**2
fig, axs = plt.subplots(1, 3, figsize=(18, 3))
axs[0].contourf(betas_quenched, Ks, mean_entropies_quenched_centered/Ks.log()[:,None], levels=100, cmap='coolwarm', vmin=0, vmax=1)
axs[0].plot(beta_c_mp, Ks, color='black', linestyle='--', label=r'$\\beta_c$')
axs[0].set_xlabel('$\\beta$')
axs[0].set_ylabel('$K$')
axs[0].set_xscale('log')
axs[0].set_title('Centered')
axs[1].contourf(betas_quenched, Ks, mean_entropies_quenched_non_centered/Ks.log()[:,None], levels=100, cmap='coolwarm', vmin=0, vmax=1)
axs[1].plot(beta_c_mp, Ks, color='black', linestyle='--', label=r'$\\beta_c$')
axs[1].set_xlabel('$\\beta$')
axs[1].set_ylabel('$K$')
axs[1].set_xscale('log')
axs[1].set_title('Raw')
axs[2].contourf(betas_quenched, Ks, mean_entropies_quenched_euclidean/Ks.log()[:,None], levels=100, cmap='coolwarm', vmin=0, vmax=1)
#axs[2].plot(beta_c_mp, Ks, color='black', linestyle='--', label=r'$\\beta_c$')
axs[2].set_xlabel('$\\beta$')
axs[2].set_ylabel('$K$')
axs[2].set_xscale('log')
axs[2].set_title('Euclidean')

for ax in axs:
    cbar_norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
    cmap = mpl.colormaps["coolwarm"]

    sm = mpl.cm.ScalarMappable(norm=cbar_norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax, label="$H/\\log(K)$")
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])

plt.savefig(f'paper_results/plots/iid_weights_quenched_entropies_N={N}.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
for K in all_Ks:
    torch.cuda.empty_cache()
    patterns = patterns_quenched_per_K[K]
    sq_norms = torch.sum(patterns**2, dim=-1)
    r_squared = sq_norms.unsqueeze(-1) + sq_norms.unsqueeze(-2) - 2 * torch.matmul(patterns, patterns.transpose(-1, -2))
    log_weight = -0.25 * torch.einsum("ckl,b->bckl", r_squared.to('cuda'), betas_quenched.to('cuda')).to('cpu')
    weights = torch.softmax(log_weight.view(log_weight.shape[0], log_weight.shape[1], -1), dim=-1)
    weights = weights.view(log_weight.shape[0], log_weight.shape[1], K, K)
    mean_part = torch.einsum("bckl,ckl->bc", weights, r_squared)
    var_part = torch.einsum("bckl,ckl->bc", weights, r_squared**2) - mean_part**2
    rescaled_metric = 1 - (betas_quenched/N).view(-1, 1) * mean_part + ((betas_quenched**2)/(N*8)).view(-1, 1)*var_part
    mean_metric = rescaled_metric.mean(dim=1)
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.plot(betas_quenched,mean_metric )
    ax.set_xscale('log')
    ax2 = ax.twinx()
    H = mean_entropies_quenched_euclidean[Ks == K,:].squeeze()
    ax2.plot(betas_quenched, H/log(K))
    ax2.axvline(betas_quenched[torch.argmin(mean_metric)], color='red', linestyle='--')
    ax2.axvline(betas_quenched[torch.argmax(mean_metric)], color='red', linestyle='--')
    plt.show()

In [ ]:
Ks.index(K, dims=0), Ks, K

# 1-step hierarchical matrices

## Generate data (dual)

In [ ]:
torch.manual_seed(1101252)
K = 256
N = 16
S = 4*K
Ms = 2**torch.arange(0, torch.log2(torch.tensor(K)).int()+1)
rho0 = 0.1
rho1 = 0.9
patterns_per_M = {}
for M in Ms:
    global_patterns = sqrt(rho0)*torch.randn(S, N)/sqrt(N)
    block_patterns = sqrt(rho1-rho0)*torch.randn(S, M, N)/sqrt(N)
    local_patterns = sqrt(1-rho1)*torch.randn(S,K, N)/sqrt(N)
    patterns = global_patterns[:,None, :] + torch.repeat_interleave(block_patterns, K//M, dim=1) + local_patterns
    patterns_per_M[M.item()] = patterns
grams_per_M = {M.item(): mhn_utils.get_gram_matrix(patterns_per_M[M.item()]) for M in Ms}
centered_grams_per_M =  {M.item(): mhn_utils.get_gram_matrix(patterns_per_M[M.item()] - patterns_per_M[M.item()].mean(dim=-2, keepdim=True)) for M in Ms}
stab_grams_per_M = {M.item(): mhn_utils.get_stability_matrix(grams_per_M[M.item()], torch.ones(K)/K) for M in Ms}
stab_centered_grams_per_M = {M.item(): mhn_utils.get_stability_matrix(centered_grams_per_M[M.item()], torch.ones(K)/K) for M in Ms}
beta_c_per_M = {M.item(): (1.0/torch.linalg.eigvalsh(stab_grams_per_M[M.item()])[:,-1]).mean() for M in Ms}
beta_c_centered_per_M = {M.item(): (1.0/torch.linalg.eigvalsh(stab_centered_grams_per_M[M.item()])[:,-1]).mean() for M in Ms}

### Quenched (dual)

In [ ]:
overwrite = False
torch.manual_seed(1101252)

num_betas = 100
num_iterations = 10_000
num_ics_per_batch = N
num_batches = 3
num_ics = num_batches * num_ics_per_batch

logit_noise_std = 1 / sqrt(N)
betas_quench = torch.logspace(-1, 2, steps=num_betas)

for M, patterns in patterns_per_M.items():
    print(f"Processing M={M}...")

    filename = (
        f"paper_results/local_data/"
        f"onestep_weights_quenched_N={N}_K={K}_M={M}_rho0={rho0:.2f}_rho1={rho1:.2f}.npz"
    )

    if Path(filename).exists() and not overwrite:
        print(f"File {filename} already exists, skipping...")
        continue

    weights_ic = torch.softmax(
        torch.randn(num_ics, K) * logit_noise_std,
        dim=-1,
    )

    results = {}

    for is_centered in [False, True]:
        for is_euclidean in [False, True]:

            # Centered and uncentered Euclidean models are equivalent
            # under a common translation of all patterns.
            if is_euclidean and is_centered:
                continue

            if device.type == "mps":
                torch.mps.empty_cache()
            elif device.type == "cuda":
                torch.cuda.empty_cache()

            patterns_used = (
                patterns - patterns.mean(dim=-2, keepdim=True)
                if is_centered
                else patterns
            )

            grams = mhn_utils.get_gram_matrix(patterns_used).to(device)

            diagonal = torch.diagonal(
                grams,
                dim1=-2,
                dim2=-1,
            )

            if is_euclidean:
                biases = -0.5 * diagonal
            else:
                biases = torch.zeros_like(diagonal)

            w_quench_fps = []

            for batch_idx in tqdm(
                range(num_batches),
                desc=(
                    f"K={K}, centered={is_centered}, "
                    f"euclidean={is_euclidean}"
                ),
            ):
                batch_start = batch_idx * num_ics_per_batch
                batch_end = (batch_idx + 1) * num_ics_per_batch

                w_batch = mhn_utils.dual_deterministic_dynamics(
                    grams,
                    biases,
                    betas_quench.to(device),
                    weights_ic[batch_start:batch_end].to(device),
                    num_iterations,
                    verbose=False,
                )

                w_quench_fps.append(w_batch.detach().cpu())

            w_quench_fps = torch.cat(w_quench_fps, dim=-2)
            entropies = mhn_utils.get_entropy(w_quench_fps)

            suffix = (
                f"centered_{is_centered}_"
                f"euclidean_{is_euclidean}"
            )

            results[f"probs_{suffix}"] = w_quench_fps.numpy()
            results[f"entropies_{suffix}"] = entropies.numpy()

    np.savez(
        filename,
        patterns=patterns.detach().cpu().numpy(),
        betas=betas_quench.numpy(),
        **results,
    )

### Annealed (dual)

In [ ]:
overwrite = False
torch.manual_seed(1101252)

num_betas = 100
num_iterations = 10_000
num_ics_per_batch = N
num_batches = 3
num_ics = num_batches * num_ics_per_batch

logit_noise_std = 1 / sqrt(N)
betas_anneal = torch.logspace(-1, 2, steps=num_betas)

for M, patterns in patterns_per_M.items():
    print(f"Processing M={M}...")

    filename = (
        f"paper_results/local_data/"
        f"onestep_weights_annealed_N={N}_K={K}_M={M}_rho0={rho0:.2f}_rho1={rho1:.2f}.npz"
    )

    if Path(filename).exists() and not overwrite:
        print(f"File {filename} already exists, skipping...")
        continue

    weights_ic = torch.softmax(
        torch.randn(num_ics, K) * logit_noise_std,
        dim=-1,
    )

    results = {}

    for is_centered in [False, True]:
        for is_euclidean in [False, True]:

            # Centered and uncentered Euclidean models are equivalent
            # under a common translation of all patterns.
            if is_euclidean and is_centered:
                continue

            if device.type == "mps":
                torch.mps.empty_cache()
            elif device.type == "cuda":
                torch.cuda.empty_cache()

            patterns_used = (
                patterns - patterns.mean(dim=-2, keepdim=True)
                if is_centered
                else patterns
            )

            grams = mhn_utils.get_gram_matrix(patterns_used)

            diagonal = torch.diagonal(
                grams,
                dim1=-2,
                dim2=-1,
            )

            if is_euclidean:
                biases = -0.5 * diagonal
            else:
                biases = torch.zeros_like(diagonal)

        
            w_anneal_fps = mhn_utils.dual_deterministic_dynamics_annealing(grams.to(device),
                                                                biases.to(device),
                                                                betas_anneal.to(device),
                                                                weights_ic.to(device),
                                                                logit_noise_std=logit_noise_std,
                                                                num_iterations=num_iterations,
                                                                verbose=True).cpu()


            entropies = mhn_utils.get_entropy(w_anneal_fps)

            suffix = (
                f"centered_{is_centered}_"
                f"euclidean_{is_euclidean}"
            )

            results[f"probs_{suffix}"] = w_anneal_fps.numpy()
            results[f"entropies_{suffix}"] = entropies.numpy()

    np.savez(
        filename,
        patterns=patterns.detach().cpu().numpy(),
        betas=betas_quench.numpy(),
        **results,
    )

## Plots quenched (dual)

In [ ]:
data_folder = Path('paper_results/local_data/')
N = 16
K = 256
rho0 = 0.1
rho1 = 0.9
betas_quenched_per_M = {}
entropies_quenched_centered_per_M = {}
entropies_quenched_non_centered_per_M = {}
entropies_quenched_euclidean_per_M  = {}
betas_quenched = None
for quenched_file in tqdm(list(data_folder.glob(f'onestep_weights_quenched_N={N}_K={K}_M=*_rho0={rho0:.2f}_rho1={rho1:.2f}.npz')), desc='Loading data...'):
    data = np.load(quenched_file, allow_pickle=True)
    M = int(quenched_file.stem.split('M=')[-1].split('_')[0])
    entropies_quenched_centered_per_M[M] = torch.as_tensor(data['entropies_centered_True_euclidean_False'])
    entropies_quenched_non_centered_per_M[M] = torch.as_tensor(data['entropies_centered_False_euclidean_False'])
    entropies_quenched_euclidean_per_M[M] = torch.as_tensor(data['entropies_centered_False_euclidean_True'])
    betas_quenched = torch.as_tensor(data['betas'])
    patterns_test = data['patterns']
    weights_test = data['probs_centered_False_euclidean_True']
    

Loading data...:   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
weights_test.shape, patterns_test.shape

((100, 512, 48), (512, 256, 16))

In [ ]:
all_Ms = list(sorted(entropies_quenched_centered_per_M.keys()))
mean_entropies_quenched_centered = []
mean_entropies_quenched_non_centered = []
mean_entropies_quenched_euclidean = []
for M in all_Ms:
    mean_entropies_quenched_centered.append(entropies_quenched_centered_per_M[M].mean(dim=(1,2)))
    mean_entropies_quenched_non_centered.append(entropies_quenched_non_centered_per_M[M].mean(dim=(1,2)))
    mean_entropies_quenched_euclidean.append(entropies_quenched_euclidean_per_M[M].mean(dim=(1,2)))
mean_entropies_quenched_centered = torch.stack(mean_entropies_quenched_centered)
mean_entropies_quenched_non_centered = torch.stack(mean_entropies_quenched_non_centered)
mean_entropies_quenched_euclidean = torch.stack(mean_entropies_quenched_euclidean)
Ms = torch.tensor(all_Ms)
Bs = K//Ms
beta_c = torch.zeros(len(Ms))
beta_c_centered = torch.zeros(len(Ms))
for i, M in enumerate(Ms):
    beta_c[i] = beta_c_per_M[M.item()]
    beta_c_centered[i] = beta_c_centered_per_M[M.item()]

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 3))
axs[0].contourf(betas_quenched, Ms/K, mean_entropies_quenched_centered/log(K), levels=100, cmap='coolwarm', vmin=0, vmax=1)
axs[0].plot(beta_c_centered, Ms/K, color='black', linestyle='--', label=r'$\\beta_c$')
axs[0].set_xlim([betas_quenched[0], betas_quenched[-1]])
axs[0].set_xlabel('$\\beta$')
axs[0].set_ylabel('$M/K$')
axs[0].set_xscale('log')
axs[0].set_yscale('log')
axs[0].set_title('Centered')
axs[1].contourf(betas_quenched, Ms/K, mean_entropies_quenched_non_centered/log(K), levels=100, cmap='coolwarm', vmin=0, vmax=1)
axs[1].plot(beta_c, Ms/K, color='black', linestyle='--', label=r'$\\beta_c$')
axs[1].set_xlim([betas_quenched[0], betas_quenched[-1]])
axs[1].set_xlabel('$\\beta$')
axs[1].set_ylabel('$M/K$')
axs[1].set_xscale('log')
axs[1].set_yscale('log')
axs[1].set_title('Raw')
axs[2].contourf(betas_quenched, Ms/K, mean_entropies_quenched_euclidean/log(K), levels=100, cmap='coolwarm', vmin=0, vmax=1)
axs[2].set_xlim([betas_quenched[0], betas_quenched[-1]])
axs[2].set_xlabel('$\\beta$')
axs[2].set_ylabel('$M/K$')
axs[2].set_xscale('log')
axs[2].set_yscale('log')
axs[2].set_title('Euclidean')
for ax in axs:
    cbar_norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
    cmap = mpl.colormaps["coolwarm"]

    sm = mpl.cm.ScalarMappable(norm=cbar_norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax, label="$H/\\log(K)$")
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
plt.savefig(f'paper_results/plots/onestep_weights_quenched_entropies_N={N}_K={K}_rho0={rho0:.2f}_rho1={rho1:.2f}.pdf', bbox_inches='tight', dpi=300)
plt.show()